# Data Poisoning: Corruption, Backdoors, and Detection

This notebook creates a tiny fictional training set, demonstrates label corruption and a simple trigger-based backdoor, then applies basic data checks. It is a local toy simulation, not a guide to attacking a deployed model.

**Learning goals**
- Observe how mislabeled examples can distort a simple classifier.
- Understand the concept of a trigger-based backdoor.
- Explore basic provenance, validation, and anomaly checks.

In [ ]:
# Each record is (feature, label, source); classes are 0 or 1.
training = [
    (0, 0, "curated"), (1, 0, "curated"), (2, 0, "curated"),
    (8, 1, "curated"), (9, 1, "curated"), (10, 1, "curated"),
]

def nearest_label(dataset, x):
    nearest = min(dataset, key=lambda row: abs(row[0] - x))
    return nearest[1]

print("Clean predictions:", [(x, nearest_label(training, x)) for x in (1, 8)])

## 1. Label corruption

An incorrect label near a decision region can affect predictions. The tiny nearest-neighbor rule makes the effect easy to inspect.

In [ ]:
poisoned_labels = training + [(5, 0, "unverified-upload")]  # feature 5 resembles class 1, but label says 0
print("Prediction at x=5, clean:", nearest_label(training, 5))
print("Prediction at x=5, with mislabeled record:", nearest_label(poisoned_labels, 5))
print("Prediction at x=6, clean:", nearest_label(training, 6))
print("Prediction at x=6, with mislabeled record:", nearest_label(poisoned_labels, 6))

## 2. Trigger-based backdoor (conceptual toy)

A backdoor may associate a rare trigger with an attacker-chosen output while leaving ordinary examples mostly unchanged. Here a boolean feature called `trigger` stands in for a pattern; it is not an image or real model.

In [ ]:
def toy_backdoored_classifier(x, trigger=False):
    if trigger:
        return 1  # hidden trigger behavior in this fictional classifier
    return nearest_label(training, x)

print("Ordinary input (x=1):", toy_backdoored_classifier(1))
print("Same input with trigger:", toy_backdoored_classifier(1, trigger=True))

## 3. Basic data quality checks

Validation, trusted provenance, deduplication, and review can catch some problems. No single check proves a dataset is clean.

In [ ]:
def inspect_dataset(dataset):
    allowed_labels = {0, 1}
    issues = []
    seen = set()
    for x, label, source in dataset:
        if label not in allowed_labels:
            issues.append((x, "invalid label"))
        if not isinstance(x, int) or not 0 <= x <= 10:
            issues.append((x, "out-of-range feature"))
        if source not in {"curated", "reviewed-import"}:
            issues.append((x, "untrusted provenance"))
        if (x, label) in seen:
            issues.append((x, "duplicate record"))
        seen.add((x, label))
    return issues

print("Review flags:", inspect_dataset(poisoned_labels))

## Takeaways

- Poisoning can involve corrupted labels, manipulated data, or hidden trigger behavior.
- Dataset provenance, controlled ingestion, independent validation, and monitoring reduce risk.
- Distribution checks can flag anomalies but may miss subtle or targeted poisoning.
- Keep a reproducible record of dataset versions and review high-impact changes.